## Esempio di generazione del testo usando un modello instruction-based

In [1]:
from model_text_gen import LocalInstructGenerator
import model_text_gen as mtg
import torch 

### Modelli open weight caricati localmente

In [3]:
if __name__ == "__main__":
    cfg = mtg.GenConfig(
        model_id="mistralai/Mistral-7B-Instruct-v0.3", # "meta-llama/Meta-Llama-3.1-8B-Instruct"
        device_map="auto",
        dtype=torch.float16,
        load_in_4bit=False,  # set True if you need to fit on smaller GPU
        max_new_tokens=140,
        temperature=0.9,
        top_p=0.92,
        top_k=40,
        repetition_penalty=1.05,
        seed=1234,
        stop_sequences=["</s>"]
    )

    gen = LocalInstructGenerator(cfg)

    role = {
        "profession": "avvocato",
        "age": 36,
        "level of education": "Laurea",
        "gender": "Maschio",
        "region": "Campania"
    }

    system_prompt = (
        "Agisci come una persona reale italiana. Rispetta rigorosamente il ruolo e lo stile indicati. "
#        "Usa italiano naturale, frasi brevi, e mantieni la coerenza della voce. "
#        "Evita anglicismi gratuiti e formule ripetitive."
    )


#    instruction = "Scrivi un breve testo sul valore del lavoro di squadra durante un turno impegnativo."
    instruction = "Scrivi una breve recensione di un ristorante"

    samples = gen.generate_role_texts(
        role=role,
        instruction=instruction,
        n_samples=3
    )

    for i, s in enumerate(samples, 1):
        print(f"\n--- Sample {i} ---\n{s}\n")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


--- Sample 1 ---
Sono un avvocato di 36 anni originario della campania. Un po' tempo fa, ho fatto un'escursione a Napoli e ho trovato un ristorante fantastico, il "Da Concetto".
Il cibo è stato meraviglioso, autentico, con un tocco di creatività che non ha smesso di sorprendermi. Ho mangiato il "Pasta alla Genovese" e il "Swordfish alla griglia", entrambi perfetti. Il servizio è stato courteous e veloce. Ho avuto


--- Sample 2 ---
Sono il Signor Di Martino, avvocato a Napoli e gourmet di cuisine locale. Sono andato al Ristorante Sorrentino quest'ultima sera e mi sono delizionato con la mia cena. Il pesce spada al limone è stato splendido, fresco e aromatico, così come il primizio di spaghetti alla chitarra con il sugo di pomodoro e basilico fresco. Il dolce, un tiramisù tradizionale, ha chiesto l'ultima bocca mia. Incontra il Ristorante Sor


--- Sample 3 ---
Sono stato a pranzo al "La Buonavita", in bellezza della mia regione. Sono stato impressionato dall'atmosfera elegante e dall'

### OpenAI API (ChatGpt)

In [6]:
from openai_text_gen import OpenAIInstructGenerator
import openai_text_gen as opn
import json 

In [7]:
apikeys = "/Users/Flint/Data/apikeys/keys.json"
with open(apikeys, 'r') as inputdata:
    K = json.load(inputdata)
    api_key = K['openai']

In [9]:
cfg = opn.GenConfig(
    model="gpt-4o-mini",   # or another suitable small model per your account availability
    max_new_tokens=140,
    temperature=0.9,
    top_p=0.92,
    seed=1234,
)
gen = OpenAIInstructGenerator(cfg, api_key=api_key)

role = {
    "profession": "muratore",
    "age": 26,
    "level of education": "primaria",
    "gender": "Maschio",
    "region": "Campania"
}

system_prompt = (
    "Agisci come una persona reale italiana. Rispetta rigorosamente il ruolo e lo stile indicati. "
    # "Usa italiano naturale, frasi brevi, e mantieni la coerenza della voce. "
    # "Evita anglicismi gratuiti e formule ripetitive."
    )

instruction = "Scrivi una breve recensione di un ristorante"

samples = gen.generate_role_texts(
    role=role,
    instruction=instruction,
    n_samples=3
)

for i, s in enumerate(samples, 1):
    print(f"\n--- Sample {i} ---\n{s}\n")



--- Sample 1 ---
Di recente sono stato al ristorante "Da Michele" e ne sono rimasto molto soddisfatto. Il servizio è stato veloce e il personale davvero cordiale. Ho ordinato una pizza margherita che era eccezionale, con ingredienti freschi e un impasto leggero. L'atmosfera è accogliente, perfetta per una serata tra amici. Consiglio vivamente di provarlo se passate da queste parti!


--- Sample 2 ---
Sono stato al ristorante "Trattoria da Nonna Rosa" a Napoli e devo dire che mi è piaciuto molto. Il cibo è davvero genuino, con piatti tipici della cucina campana, come la pasta alla genovese e la pizza fritta. L'atmosfera è accogliente, ti senti subito a casa. I prezzi sono onesti e il personale è molto gentile. Un posto da consigliare a chi ama il buon cibo!


--- Sample 3 ---
Sono andato a mangiare in un ristorante a Napoli e devo dire che mi ha colpito. L'atmosfera è accogliente e il personale è molto gentile. Ho assaggiato la pizza margherita e la pasta al pomodoro: davvero squisite 

### Generazione dei profili

In [2]:
import numpy as np

In [5]:
professions = ['insegnante', 'avvocato', 'muratore', 
               'contadino', 'medico', 'infermiere', 
               'operatore socio-sanitario',
               'studente', 'disoccupato']
ages = [(8, 12), (13, 18), (19, 35), (36, 50), (51, 71), (72, 90)]
education_levels = ['primaria', 'secondaria', 'superiore', 'laurea', 'post-laurea']
gender = ['maschio', 'femmina']
region = ["Valle d'Aosta", "Piemonte", "Ligura", "Lombardia", "Veneto", "Friuli Venezia-Giulia",
          "Trentino Alto Adige", "Toscana", "Emilia Romagna", "Marche", "Umbria", "Lazio", "Abruzzo",
          "Campania", "Basilicata", "Puglia", "Calabria", "Sicilia", "Sardegna"]

rules = [
    {'age': (8, 12), 'professions': ['studente'], 'edu': ['primaria']},
    {'age': (13, 18), 'professions': ['studente', 
                                      'contadino',
                                      'muratore',
                                      'disoccupato'], 
                                      'edu': ['primaria', 'secondaria']},
    ]

def check_rules(age, profession, education, rules):
    checked = True
    for rule in rules:
        min_age, max_age = rule['age']
        if min_age <= age <= max_age:
            if profession not in rule['professions'] or education not in rule['edu']:
                checked = False
                break 
    return checked

profiles = []
counter = 0
for profession in professions:
    for min_age, max_age in ages:
        age = np.random.choice(list(range(min_age, max_age + 1)))
        for edu in education_levels:
            check = check_rules(age=age, profession=profession, education=edu, rules=rules)
            if check:
                for gen in gender:
                    for reg in region:
                        metadata = {'id': counter, 'age_class': f"{min_age}-{max_age}"}
                        counter += 1
                        role = {
                            "profession": profession,
                            "age": f"{age}",
                            "level of education": edu,
                            "gender": gen,
                            "region": reg
                        }
                        profiles.append({'metadata': metadata, 'data': role})
            else:
                pass
print(f"Abbiamo generato {len(profiles)} profili")

Abbiamo generato 7182 profili


In [6]:
np.random.shuffle(profiles)
profiles[0]

{'metadata': {'id': 6898, 'age_class': '51-71'},
 'data': {'profession': 'disoccupato',
  'age': '63',
  'level of education': 'superiore',
  'gender': 'femmina',
  'region': 'Piemonte'}}

In [7]:
import json

In [9]:
file_output = "./data/profiles.json"
with open(file_output, 'w') as out:
    json.dump({'profiles': profiles}, out)